# Hypothyroidism T4 Non-Responder Cohort — All of Us (v2)

Identifies hypothyroid patients and classifies their response to levothyroxine (T4),
building toward a feasibility assessment for a genetic study of **non-responders**.

This notebook is a **thin orchestrator**: all cohort logic lives in the importable
`cohort` package (`config`, `concepts`, `ctes`, `queries`). See `PLAN.md` for the
protocol of record and `README.md` for the design.

**Refinements over v1** (phenotype precision for a cleaner non-responder definition):
- ≥ 2 hypothyroid diagnoses on separate dates
- adult at first T4; elevated pre-treatment TSH confirming disease
- **exclusions** for mimics of non-response (malabsorption, central hypothyroidism,
  thyroid cancer, pregnancy; optional interfering/thyroid-disrupting drugs)
- response classified from **sustained** on-treatment TSH (≥ 2 measurements)

**Deliberately deferred** (pending data-availability checks): weight-based T4 dose
adequacy, and GWAS-specific criteria (ancestry/PCs/association) — see `PLAN.md`.


## 1. Setup


In [ ]:
import os, sys

# Make the cohort package importable when the notebook runs from v2/
sys.path.insert(0, os.path.abspath("."))

import pandas as pd
from cohort import CohortConfig, Runner, concepts, queries

runner = Runner()                       # uses WORKSPACE_CDR + ambient creds
cfg = CohortConfig(dataset=runner.dataset)
# cfg.enable_drug_exclusions = True     # optional: add interfering/disrupting-drug exclusions
print("CDR dataset:", cfg.dataset)

## 2. Concept discovery

Concept IDs are resolved from the CDR vocabulary (not hardcoded) and stored on `cfg`.
Review each result before relying on it downstream.


In [ ]:
roots = concepts.discover_hypo_roots(runner, cfg)
print("Hypothyroidism root SNOMED concepts:", cfg.hypo_root_concept_ids)
roots

In [ ]:
cfg.t4_ingredient_id = concepts.discover_ingredient(runner, cfg, "levothyroxine")
cfg.t3_ingredient_id = concepts.discover_ingredient(runner, cfg, "liothyronine")
tsh = concepts.discover_tsh_concepts(runner, cfg)
print("T4 ingredient:", cfg.t4_ingredient_id, "| T3 ingredient:", cfg.t3_ingredient_id)
print("TSH concept IDs:", cfg.tsh_concept_ids)
tsh

In [ ]:
# Thyroid lab panel (data-availability discovery by LOINC name). Review the mapping.
labs = concepts.discover_lab_panel(runner, cfg)
print({k: len(v) for k, v in cfg.lab_concept_ids.items()})
labs

In [ ]:
# Or resolve everything at once (also discovers drug-exclusion ingredients if enabled):
# concepts.resolve(runner, cfg)

## 3. Base cohort

Hypothyroid dx (≥2) + adult + T4 within the eligibility window after diagnosis +
elevated pre-treatment TSH + an on-treatment TSH available at steady state +
T3 not co-started + not excluded by a confounding condition.


In [ ]:
df_base = runner.df(queries.base_cohort_query(cfg))
print(f"Base cohort size: {len(df_base):,}")
df_base.head()

## 4. Base cohort — TSH response classification

`non_responder` = ≥ 2 on-treatment TSH, none normalized, ≥ 2 elevated.
`responder` = ≥ 2 normalized. Otherwise `partial` / `insufficient_data`.


In [ ]:
df_cls = runner.df(queries.base_tsh_classification_query(cfg))
print(df_cls["response_class"].value_counts())
df_cls.head()

## 5. Base cohort — T3 add-on non-responders

Liothyronine (T3) started after the co-start window but within the add-on window —
a clinician escalating off T4 monotherapy is a treatment-failure signal.


In [ ]:
df_t3 = runner.df(queries.base_t3_nonresponder_query(cfg))
print(f"T3 add-on non-responders: {len(df_t3):,}")
df_t3.head()

## 6. EHR symptom cohort

Base cohort + ≥ 1 target symptom (fatigue / constipation / brain fog) in EHR before
diagnosis, with its own TSH, T3, and symptom-persistence response queries.

Symptom-persistence response compares each member's pre-T4 symptom categories to a
post-T4 window: `symptom_responder` (pre-T4 symptoms, none recur), `symptom_nonresponder`
(≥ 1 pre-T4 symptom recurs), or `no_pre_symptom`. A post-T4 *absence* reflects the lack of
a coded encounter, not necessarily true resolution — weaker evidence than TSH.


In [ ]:
df_ehr = runner.df(queries.ehr_symptom_cohort_query(cfg))
print(f"EHR symptom cohort size: {len(df_ehr):,}")

df_ehr_cls = runner.df(queries.ehr_tsh_classification_query(cfg))
print(df_ehr_cls["response_class"].value_counts())

# Symptom-persistence classification: responder (symptoms resolve) vs.
# non-responder (symptoms persist) vs. no pre-T4 symptom in the window.
df_ehr_sym = runner.df(queries.ehr_symptom_classification_query(cfg))
print(df_ehr_sym["symptom_response_class"].value_counts())

## 7. Survey (PPI) symptom cohort

Base cohort + ≥ 1 target symptom reported in PPI survey observations before diagnosis.

Because PPI answers are coded (`observation.value_as_concept_id`), the survey path reads
the **answer value**, not just the presence of a question. It adds a positively-confirmed
responder class: `symptom_responder_confirmed` — a post-T4 answer affirming the symptom is
gone (e.g. "Not at all"; answer concepts in `PPI_ABSENT_ANSWER_KEYWORDS` — **review them**).
A "gone" answer is excluded from persistence, so answering "Not at all" post-T4 no longer
counts as the symptom persisting. `symptom_responder` remains the weaker, inferred-by-absence
class (no post-T4 record at all).


In [ ]:
df_srv = runner.df(queries.survey_symptom_cohort_query(cfg))
print(f"Survey symptom cohort size: {len(df_srv):,}")

df_srv_cls = runner.df(queries.survey_tsh_classification_query(cfg))
print(df_srv_cls["response_class"].value_counts())

# Symptom-persistence classification: responder (symptoms resolve) vs.
# non-responder (symptoms persist) vs. no pre-T4 symptom in the window.
df_srv_sym = runner.df(queries.survey_symptom_classification_query(cfg))
print(df_srv_sym["symptom_response_class"].value_counts())

## 8. Feasibility — lab & genomic data availability

How many base-cohort members have each thyroid lab, and how many have genomic data.
The genomic count determines whether a downstream genetic analysis is feasible at all.


In [ ]:
runner.df(queries.lab_availability_query(cfg))

In [ ]:
# Requires the cb_search_person helper table (All of Us).
runner.df(queries.genomic_availability_query(cfg))